# 5. Compute dynamic features

Versione pulita: le parti dati restano in `data/processed/features`; i grafici esplorativi vanno in `reports/03_dynamic_features/exploratory_figures`, quelli da paper in `reports/03_dynamic_features/paper_figures`.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from the current working directory."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        "Project root not found. Run the notebook from inside the project repository."
    )


def ensure_dir(path: Path) -> Path:
    """Create a directory if needed and return it."""
    path.mkdir(parents=True, exist_ok=True)
    return path


def report_dir(root: Path, analysis_name: str, subfolder: str | None = None) -> Path:
    """Create and return a report directory for one analysis."""
    base = root / "reports" / analysis_name
    return ensure_dir(base if subfolder is None else base / subfolder)


ROOT = find_project_root()
DATA_FEATURES_DIR = ensure_dir(ROOT / "data" / "processed" / "features")
REPORT_DYNAMIC_DIR = report_dir(ROOT, "03_dynamic_features")
EXPLORATORY_FIG_DIR = report_dir(ROOT, "03_dynamic_features", "exploratory_figures")
PAPER_FIG_DIR = report_dir(ROOT, "03_dynamic_features", "paper_figures")

print("ROOT:", ROOT)


ROOT: /home/eleonorab/repository/convective_cooling


In [2]:
# =========================
# Build dynamic analysis dataset
# =========================

import pandas as pd
import numpy as np
from scipy.signal import savgol_filter

INPUT_DIR = ROOT / "data" / "processed" / "aggregated_by_signal"
OUTPUT_FILE = DATA_FEATURES_DIR / "analysis_dataset_dynamic.csv"

BIN_SECONDS = 10

SIGNAL_FILES = {
    "t_dorsal": "t_dorsal.csv",
    "t_microclimateback": "t_microclimateback.csv",
}

print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_FILE:", OUTPUT_FILE)


def assign_phase(elapsed_min: float) -> str:
    if elapsed_min < 10:
        return "standing_1"
    if elapsed_min < 20:
        return "walk_low_1"
    if elapsed_min < 30:
        return "walk_mod_1"
    if elapsed_min < 40:
        return "walk_low_2"
    if elapsed_min < 50:
        return "walk_mod_2"
    return "standing_2"


def require_file(path: Path, message: str | None = None) -> None:
    if not path.exists():
        available = sorted(p.name for p in path.parent.glob("*.csv")) if path.parent.exists() else []
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            f"{message or ''}\n"
            f"Available CSV files in {path.parent}: {available}"
        )


def load_signal(signal_file: str, value_name: str) -> pd.DataFrame:
    path = INPUT_DIR / signal_file
    require_file(
        path,
        "Run 1_make_dataset.ipynb first and check that SIGNAL_GROUPS uses the expected signal names."
    )

    df = pd.read_csv(path)

    required_cols = ["subject_id", "condition", "elapsed_seconds", "elapsed_minutes", "value"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df[required_cols].copy()
    df["subject_id"] = df["subject_id"].astype(str).str.strip()
    df["condition"] = df["condition"].astype(str).str.strip()
    df["elapsed_seconds"] = pd.to_numeric(df["elapsed_seconds"], errors="coerce")
    df["elapsed_minutes"] = pd.to_numeric(df["elapsed_minutes"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.dropna(subset=["subject_id", "condition", "elapsed_seconds", "value"]).copy()

    df["elapsed_seconds_bin"] = (df["elapsed_seconds"] / BIN_SECONDS).round() * BIN_SECONDS
    df["elapsed_minutes_bin"] = df["elapsed_seconds_bin"] / 60.0

    df = (
        df.groupby(
            ["subject_id", "condition", "elapsed_seconds_bin", "elapsed_minutes_bin"],
            as_index=False,
        )["value"]
        .mean()
    )

    return df.rename(columns={"value": value_name})


# Load and merge the two signals needed for the local gradient.
dorsal = load_signal(SIGNAL_FILES["t_dorsal"], "t_dorsal")
micro = load_signal(SIGNAL_FILES["t_microclimateback"], "t_microclimateback")

df = pd.merge(
    dorsal,
    micro,
    on=["subject_id", "condition", "elapsed_seconds_bin", "elapsed_minutes_bin"],
    how="inner",
)

if df.empty:
    raise ValueError(
        "The merged dynamic dataframe is empty. Check that t_dorsal and "
        "t_microclimateback have overlapping subject/condition/time bins."
    )

df["phase"] = df["elapsed_minutes_bin"].apply(assign_phase)
df["delta_t_loc"] = df["t_dorsal"] - df["t_microclimateback"]

print("Merged dynamic dataframe shape:", df.shape)
print("Rows by condition:")
print(df["condition"].value_counts(dropna=False))


def compute_dynamic_group(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("elapsed_seconds_bin").copy()

    y = g["t_dorsal"].to_numpy(dtype=float)
    t = g["elapsed_seconds_bin"].to_numpy(dtype=float)
    mask = np.isfinite(y) & np.isfinite(t)

    g["t_dorsal_smooth"] = np.nan
    g["dt_dorsal_dt"] = np.nan
    g["minus_dt_dorsal_dt"] = np.nan

    if mask.sum() < 15:
        return g

    y_valid = y[mask]
    t_valid = t[mask]

    window = 11 if len(y_valid) >= 11 else len(y_valid) // 2 * 2 + 1
    if window < 5:
        return g

    y_smooth = savgol_filter(y_valid, window_length=window, polyorder=2)
    derivative = np.gradient(y_smooth) / np.gradient(t_valid)

    g.loc[mask, "t_dorsal_smooth"] = y_smooth
    g.loc[mask, "dt_dorsal_dt"] = derivative
    g.loc[mask, "minus_dt_dorsal_dt"] = -derivative

    return g


df = (
    df.groupby(["subject_id", "condition"], group_keys=False)
    .apply(compute_dynamic_group)
    .reset_index(drop=True)
)

# Apparent heat-transfer index. Values can be unstable when cooling rate is close to zero.
df["hti_app"] = df["delta_t_loc"] / df["minus_dt_dorsal_dt"]
df.replace([np.inf, -np.inf], np.nan, inplace=True)

ensure_dir(OUTPUT_FILE.parent)
df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Non-null rows by condition:")
print(df.groupby("condition")[["delta_t_loc", "minus_dt_dorsal_dt", "hti_app"]].count())


INPUT_DIR: /home/eleonorab/repository/convective_cooling/data/processed/aggregated_by_signal
OUTPUT_FILE: /home/eleonorab/repository/convective_cooling/data/processed/features/analysis_dataset_dynamic.csv
Merged dynamic dataframe shape: (8987, 8)
Rows by condition:
condition
no_wind      3238
wind_low     2875
wind_high    2874
Name: count, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/data/processed/features/analysis_dataset_dynamic.csv
Non-null rows by condition:
           delta_t_loc  minus_dt_dorsal_dt  hti_app
condition                                          
no_wind           3238                3238     2731
wind_high         2874                2874     2629
wind_low          2875                2875     2641


/tmp/ipykernel_5592/2016354157.py:142: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_dynamic_group)


In [3]:
# =========================
# Extract dynamic phase features
# =========================

import pandas as pd
import numpy as np

INPUT_FILE = DATA_FEATURES_DIR / "analysis_dataset_dynamic.csv"
OUTPUT_FILE = DATA_FEATURES_DIR / "dynamic_phase_features.csv"

require_file(INPUT_FILE, "Run the previous cell first.")
df = pd.read_csv(INPUT_FILE)

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]

SIGNALS_TO_SUMMARISE = ["delta_t_loc", "hti_app", "minus_dt_dorsal_dt"]

required_cols = ["subject_id", "condition", "phase", "elapsed_minutes_bin", *SIGNALS_TO_SUMMARISE]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {INPUT_FILE}: {missing}")

df["subject_id"] = df["subject_id"].astype(str).str.strip()
df["condition"] = df["condition"].astype(str).str.strip()
df["phase"] = df["phase"].astype(str).str.strip()

for col in ["elapsed_minutes_bin", "elapsed_seconds_bin", *SIGNALS_TO_SUMMARISE]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df.replace([np.inf, -np.inf], np.nan, inplace=True)


def compute_auc(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 2:
        return np.nan
    return np.trapz(y, x)


rows = []
for (subj, cond, phase), g in df.groupby(["subject_id", "condition", "phase"], observed=True):
    g = g.sort_values("elapsed_minutes_bin").copy()

    for signal in SIGNALS_TO_SUMMARISE:
        valid = (
            g[["elapsed_minutes_bin", signal]]
            .rename(columns={"elapsed_minutes_bin": "x", signal: "y"})
            .dropna()
        )

        if valid.empty:
            continue

        last_2min = valid[valid["x"] >= (valid["x"].max() - 2)]

        rows.append({
            "subject_id": subj,
            "condition": cond,
            "phase": phase,
            "signal": signal,
            "n_samples": len(valid),
            "mean": valid["y"].mean(),
            "median": valid["y"].median(),
            "p25": valid["y"].quantile(0.25),
            "p75": valid["y"].quantile(0.75),
            "mean_last2min": last_2min["y"].mean() if not last_2min.empty else np.nan,
            "auc": compute_auc(valid["x"].to_numpy(dtype=float), valid["y"].to_numpy(dtype=float)),
        })

df_feat = pd.DataFrame(rows)

if not df_feat.empty:
    df_feat["phase"] = pd.Categorical(df_feat["phase"], categories=PHASE_ORDER, ordered=True)
    df_feat = df_feat.sort_values(["signal", "condition", "subject_id", "phase"]).reset_index(drop=True)

ensure_dir(OUTPUT_FILE.parent)
df_feat.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Shape:", df_feat.shape)
print(df_feat.head())

print("\nRows by signal/condition:")
if not df_feat.empty:
    print(df_feat.groupby(["signal", "condition"]).size())


Saved: /home/eleonorab/repository/convective_cooling/data/processed/features/dynamic_phase_features.csv
Shape: (486, 11)
  subject_id condition       phase       signal  n_samples      mean  median  \
0  tester_01   no_wind  standing_1  delta_t_loc         60 -1.115100  -1.203   
1  tester_01   no_wind  walk_low_1  delta_t_loc         60 -0.356283  -0.518   
2  tester_01   no_wind  walk_mod_1  delta_t_loc         60  0.230933   0.200   
3  tester_01   no_wind  walk_low_2  delta_t_loc         60  0.276850   0.294   
4  tester_01   no_wind  walk_mod_2  delta_t_loc         60  0.469100   0.449   

       p25      p75  mean_last2min        auc  
0 -1.39000 -0.89100      -1.151583 -10.997167  
1 -0.65825 -0.04975       0.298385  -3.523417  
2  0.16800  0.29400       0.158769   2.265667  
3  0.16900  0.35600       0.274308   2.724833  
4  0.41800  0.54300       0.572077   4.610833  

Rows by signal/condition:
signal              condition
delta_t_loc         no_wind      60
                 

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = find_project_root()
INPUT_FILE = ROOT / "data" / "processed" / "features" / "dynamic_phase_features.csv"
OUTPUT_DIR = report_dir(ROOT, "03_dynamic_features", "exploratory_figures")
ensure_dir(OUTPUT_DIR)

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]
CONDITION_ORDER = ["no_wind", "wind_low", "wind_high"]

df_feat = pd.read_csv(INPUT_FILE)
df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

df_feat["condition"] = df_feat["condition"].astype(str).str.strip()
df_feat["phase"] = df_feat["phase"].astype(str).str.strip()
df_feat["signal"] = df_feat["signal"].astype(str).str.strip()

for col in ["mean", "median", "p25", "p75", "mean_last2min", "auc"]:
    if col in df_feat.columns:
        df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")


def plot_phase_feature(df, signal, metric, filename):
    sub = df[df["signal"] == signal].copy()
    sub = sub[np.isfinite(sub[metric])].copy()

    print(f"\n{signal} | {metric} | rows:", len(sub))
    print(sub.groupby(["condition", "phase"])[metric].count())

    if sub.empty:
        print(f"Skipping {signal} - {metric}: no valid data")
        return

    plot_df = (
        sub.groupby(["condition", "phase"], observed=True)[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    plot_df["sem"] = plot_df["std"] / np.sqrt(plot_df["count"])
    phase_to_x = {phase: i for i, phase in enumerate(PHASE_ORDER)}

    plt.figure(figsize=(8, 5))
    plotted = 0

    for condition in CONDITION_ORDER:
        cond_df = plot_df[plot_df["condition"] == condition].copy()
        if cond_df.empty:
            continue

        cond_df["phase"] = pd.Categorical(cond_df["phase"], categories=PHASE_ORDER, ordered=True)
        cond_df = cond_df.sort_values("phase")

        x = [phase_to_x[p] for p in cond_df["phase"].astype(str)]
        y = cond_df["mean"].to_numpy(dtype=float)
        sem = cond_df["sem"].fillna(0).to_numpy(dtype=float)

        plt.plot(x, y, marker="o", label=condition)
        plt.fill_between(x, y - sem, y + sem, alpha=0.2)
        plotted += 1

    if plotted == 0:
        print(f"Skipping {signal} - {metric}: nothing plotted")
        plt.close()
        return

    plt.xticks(list(phase_to_x.values()), list(phase_to_x.keys()), rotation=45)
    plt.xlabel("Phase", fontsize=12)
    plt.ylabel(f"{signal} - {metric}", fontsize=12)
    plt.title(f"{signal}: {metric} by phase and condition", fontsize=14)
    plt.legend()
    plt.tight_layout()

    out_path = OUTPUT_DIR / filename
    plt.savefig(out_path, dpi=300)
    plt.close()
    print("Saved:", out_path)


plot_phase_feature(df_feat, "delta_t_loc", "mean", "delta_t_loc_mean_by_phase.png")
plot_phase_feature(df_feat, "minus_dt_dorsal_dt", "mean", "minus_dt_dorsal_dt_mean_by_phase.png")
plot_phase_feature(df_feat, "hti_app", "mean", "hti_app_mean_by_phase.png")
plot_phase_feature(df_feat, "hti_app", "mean_last2min", "hti_app_mean_last2min_by_phase.png")
plot_phase_feature(df_feat, "delta_t_loc", "auc", "delta_t_loc_auc_by_phase.png")


delta_t_loc | mean | rows: 162
condition  phase     
no_wind    standing_1    10
           standing_2    10
           walk_low_1    10
           walk_low_2    10
           walk_mod_1    10
           walk_mod_2    10
wind_high  standing_1     8
           standing_2     8
           walk_low_1     8
           walk_low_2     8
           walk_mod_1     8
           walk_mod_2     8
wind_low   standing_1     9
           standing_2     9
           walk_low_1     9
           walk_low_2     9
           walk_mod_1     9
           walk_mod_2     9
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/03_dynamic_features/exploratory_figures/delta_t_loc_mean_by_phase.png

minus_dt_dorsal_dt | mean | rows: 162
condition  phase     
no_wind    standing_1    10
           standing_2    10
           walk_low_1    10
           walk_low_2    10
           walk_mod_1    10
           walk_mod_2    10
wind_high  standing_1     8
           standing_2     8
 

In [5]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = find_project_root()
INPUT_FILE = ROOT / "data" / "processed" / "features" / "dynamic_phase_features.csv"
OUTPUT_DIR = report_dir(ROOT, "03_dynamic_features", "exploratory_figures")
ensure_dir(OUTPUT_DIR)

df_feat = pd.read_csv(INPUT_FILE)
df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

df_feat["condition"] = df_feat["condition"].astype(str).str.strip()
df_feat["phase"] = df_feat["phase"].astype(str).str.strip()
df_feat["signal"] = df_feat["signal"].astype(str).str.strip()

for col in ["mean", "mean_last2min", "auc"]:
    if col in df_feat.columns:
        df_feat[col] = pd.to_numeric(df_feat[col], errors="coerce")

PHASES_OF_INTEREST = ["walk_mod_1", "walk_mod_2", "standing_2"]
CONDITION_ORDER = ["no_wind", "wind_low", "wind_high"]


def plot_condition_distribution(df, signal, metric, phases, prefix):
    sub = df[
        (df["signal"] == signal) &
        (df["phase"].isin(phases))
    ].copy()

    for phase in phases:
        phase_df = sub[sub["phase"] == phase].copy()
        phase_df = phase_df[np.isfinite(phase_df[metric])].copy()

        print(f"\n{signal} | {metric} | {phase}")
        if phase_df.empty:
            print("No valid data")
            continue

        print(phase_df.groupby("condition")[metric].count())

        plt.figure(figsize=(7, 5))

        sns.violinplot(
            data=phase_df,
            x="condition",
            y=metric,
            order=CONDITION_ORDER,
            inner=None,
            cut=0
        )

        sns.stripplot(
            data=phase_df,
            x="condition",
            y=metric,
            order=CONDITION_ORDER,
            dodge=False,
            alpha=0.8
        )

        plt.xlabel("Condition", fontsize=12)
        plt.ylabel(f"{signal} - {metric}", fontsize=12)
        plt.title(f"{signal}: {metric} by condition - {phase}", fontsize=14)
        plt.tight_layout()

        out_path = OUTPUT_DIR / f"{prefix}_{signal}_{metric}_{phase}.png"
        plt.savefig(out_path, dpi=300)
        plt.close()

        print("Saved:", out_path)


plot_condition_distribution(df_feat, "delta_t_loc", "mean", PHASES_OF_INTEREST, "condition_distribution")
plot_condition_distribution(df_feat, "minus_dt_dorsal_dt", "mean", PHASES_OF_INTEREST, "condition_distribution")
plot_condition_distribution(df_feat, "hti_app", "mean", PHASES_OF_INTEREST, "condition_distribution")


delta_t_loc | mean | walk_mod_1
condition
no_wind      10
wind_high     8
wind_low      9
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/03_dynamic_features/exploratory_figures/condition_distribution_delta_t_loc_mean_walk_mod_1.png

delta_t_loc | mean | walk_mod_2
condition
no_wind      10
wind_high     8
wind_low      9
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/03_dynamic_features/exploratory_figures/condition_distribution_delta_t_loc_mean_walk_mod_2.png

delta_t_loc | mean | standing_2
condition
no_wind      10
wind_high     8
wind_low      9
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/03_dynamic_features/exploratory_figures/condition_distribution_delta_t_loc_mean_standing_2.png

minus_dt_dorsal_dt | mean | walk_mod_1
condition
no_wind      10
wind_high     8
wind_low      9
Name: mean, dtype: int64
Saved: /home/eleonorab/repository/convective_cooling/reports/